In [ ]:
# # Environment Setup
# import os
# os.environ["CC"] = os.path.expanduser("~/gcc_wrapper")
# os.environ.pop("GCC_EXEC_PREFIX", None)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"


# M1 Combined Dataset in local

m1_combined was pretty good, with roughly 7.5k samples from m1 including mcq options in prompt and 10k from juicy mix and a 10% eval set.

trained one epoch for dpo and tested in dpo, mcqa tasks with also tensorboard logging so now we have a much better idea of how the model is doing, how to improve and how to create a proper dataset.



In [3]:
# Clean imports - data only
import datetime
from datasets import load_dataset, Dataset, DatasetDict

# Single, clean data pipeline function
def create_dpo_datasets():
    """
    Create DPO datasets with proper splits for training
    - Creates both M1-only and combined datasets
    - Larger combined dataset with 30k juicy mix samples
    - 80/10/10 train/eval/test splits
    """
    
    # 1. Load datasets
    print("Loading datasets...")
    
    m1_dpo = load_dataset("ciacco/m1_dpo_filtered_highest_total", split="train")
    print(f"M1 dataset: {len(m1_dpo)} samples")
    
    juicy_mix = load_dataset("ciacco/m1_dpo_juicy_mix", split="train")
    print(f"Juicy mix dataset: {len(juicy_mix)} samples")
    
    # 2. Standardize M1 dataset format
    def standardize_m1_sample(sample):
        prompt = sample['prompt']
        if sample.get('question_type') == 'mcq' and sample.get('question_options'):
            if "Options:" not in prompt and sample['question_options']:
                prompt = f"{prompt}\n\nOptions:\n{sample['question_options']}"
        
        return {
            'prompt': prompt,
            'chosen': sample['chosen'],
            'rejected': sample['rejected'],
            'source': 'm1_dataset'
        }
    
    m1_standardized = m1_dpo.map(standardize_m1_sample)
    m1_standardized = m1_standardized.select_columns(['prompt', 'chosen', 'rejected', 'source'])
    
    # 3. Sample from juicy mix - LARGE version (30k)
    juicy_sample_size = min(30000, len(juicy_mix))
    juicy_sampled = juicy_mix.shuffle(seed=42).select(range(juicy_sample_size))
    
    if 'source' not in juicy_sampled.column_names:
        juicy_sampled = juicy_sampled.add_column('source', ['juicy_mix'] * len(juicy_sampled))
    
    print(f"Sampled juicy mix: {len(juicy_sampled)} samples")
    
    # 4. Create datasets
    # M1 only (for domain-specific training)
    m1_only = m1_standardized
    
    # Combined dataset (M1 + large juicy sample)
    combined_list = list(m1_standardized) + list(juicy_sampled)
    combined_dataset = Dataset.from_list(combined_list)
    
    print(f"M1 only: {len(m1_only)} samples")
    print(f"Combined: {len(combined_dataset)} samples")
    
    # 5. Create train/eval/test splits (80/10/10)
    def create_three_way_splits(dataset, eval_size=0.1, test_size=0.1, seed=42):
        dataset = dataset.shuffle(seed=seed)
        
        test_split_idx = int(len(dataset) * test_size)
        eval_split_idx = int(len(dataset) * (test_size + eval_size))
        
        test_dataset = dataset.select(range(test_split_idx))
        eval_dataset = dataset.select(range(test_split_idx, eval_split_idx))
        train_dataset = dataset.select(range(eval_split_idx, len(dataset)))
        
        return DatasetDict({
            'train': train_dataset,
            'eval': eval_dataset,
            'test': test_dataset
        })
    
    # Create splits for both datasets
    m1_splits = create_three_way_splits(m1_only)
    combined_splits = create_three_way_splits(combined_dataset)
    
    print(f"\nDataset splits:")
    print(f"M1 only - Train: {len(m1_splits['train'])}, Eval: {len(m1_splits['eval'])}, Test: {len(m1_splits['test'])}")
    print(f"Combined - Train: {len(combined_splits['train'])}, Eval: {len(combined_splits['eval'])}, Test: {len(combined_splits['test'])}")
    
    # 6. Push to hub
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    
    m1_dataset_name = f"ciacco/m1_dpo_only_{timestamp}"
    combined_dataset_name = f"ciacco/m1_dpo_large_combined_{timestamp}"
    
    m1_splits.push_to_hub(m1_dataset_name, private=True)
    combined_splits.push_to_hub(combined_dataset_name, private=True)
    
    print(f"\nDatasets pushed to hub:")
    print(f"M1 only: {m1_dataset_name}")
    print(f"Combined: {combined_dataset_name}")
    
    return {
        'm1_only': (m1_splits, m1_dataset_name),
        'combined': (combined_splits, combined_dataset_name),
        'timestamp': timestamp
    }


In [4]:

# Run Data Pipeline
print("Creating DPO datasets...")
datasets = create_dpo_datasets()

print(f"\n✅ Data pipeline completed!")
print(f"📊 Use these dataset names for training:")
print(f"   • M1 only: {datasets['m1_only'][1]}")
print(f"   • Combined: {datasets['combined'][1]}")

Creating DPO datasets...
Loading datasets...


README.md:   0%|          | 0.00/683 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7584 [00:00<?, ? examples/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]


Datasets pushed to hub:
M1 only: ciacco/m1_dpo_only_20250609_0508
Combined: ciacco/m1_dpo_large_combined_20250609_0508

✅ Data pipeline completed!
📊 Use these dataset names for training:
   • M1 only: ciacco/m1_dpo_only_20250609_0508
   • Combined: ciacco/m1_dpo_large_combined_20250609_0508


---

In [ ]:

# Unsloth DPO Training Setup
def setup_unsloth_dpo_training(dataset_name, model_name="Qwen/Qwen-0.6B-Base"):
    """
    Setup Unsloth DPO training configuration
    """
    
    # Load model and tokenizer with Unsloth
    model, tokenizer = unsloth.FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=2048,
        dtype=None,  # Auto-detect
        load_in_4bit=True,
    )
    
    # Add LoRA adapters
    model = unsloth.FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    
    # Load dataset
    dataset = load_dataset(dataset_name)
    
    # DPO Training configuration
    training_args = DPOConfig(
        output_dir=f"./results/dpo_unsloth_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}",
        num_train_epochs=2,  # 2 epochs as requested
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,  # Lower LR for DPO
        lr_scheduler_type="cosine",  # Cosine decay as requested
        warmup_ratio=0.05,  # Shorter warmup (5% instead of 10%)
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="tensorboard",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
        bf16=True,  # Use bf16 for better stability
    )
    
    # Initialize DPO trainer
    trainer = DPOTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["eval"],
        tokenizer=tokenizer,
        beta=0.1,  # DPO beta parameter
        max_length=2048,
        max_prompt_length=1024,
    )
    
    return trainer, model, tokenizer

def run_dpo_training(trainer, model_name_suffix="unsloth_dpo_2epoch"):
    """
    Run DPO training and push model to hub
    """
    print("Starting DPO training...")
    
    # Train
    trainer.train()
    
    # Save final model
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    final_model_name = f"ciacco/{model_name_suffix}_{timestamp}"
    
    trainer.save_model()
    trainer.model.push_to_hub(final_model_name, private=True)
    trainer.tokenizer.push_to_hub(final_model_name, private=True)
    
    print(f"Model saved and pushed to: {final_model_name}")
    
    return final_model_name

In [ ]:
# ...existing code...

# Add missing import
#from unsloth import FastLanguageModel

# ...existing code...

# Training Setup (for cluster/hub execution)
# Uncomment and run this section when on cluster with GPU

# # Create directories
# import os
# os.makedirs("./outputs", exist_ok=True)
# os.makedirs("./logs", exist_ok=True)

# # Setup and run training
# trainer, model, tokenizer = setup_unsloth_dpo_training(dataset_name)
# final_model_name, lora_model_name = run_dpo_training(trainer)

# print(f"Training completed!")
# print(f"Merged model: {final_model_name}")
# print(f"LoRA adapters: {lora_model_name}")

# # Optional: Resume from checkpoint if needed
# # checkpoint_path = "./outputs/dpo_unsloth_YYYYMMDD_HHMM/checkpoint-100"
# # trainer = resume_training_from_checkpoint(checkpoint_path, dataset_name)

----

In [ ]:



from trl import DPOTrainer, DPOConfig
import unsloth


ModuleNotFoundError: No module named 'unsloth'

In [ ]:

# Data Pipeline Function
def create_clean_dpo_pipeline():
    """
    Create a clean DPO training pipeline with proper splits and filtering
    """
    
    # 1. Load and combine datasets
    print("Loading datasets...")
    
    # Load your M1 dataset (smaller, domain-specific)
    m1_dpo = load_dataset("ciacco/m1_dpo_filtered_highest_total", split="train")
    print(f"M1 dataset: {len(m1_dpo)} samples")
    
    # Load juicy mix (larger, general)
    juicy_mix = load_dataset("ciacco/m1_dpo_juicy_mix", split="train")
    print(f"Juicy mix dataset: {len(juicy_mix)} samples")
    
    # 2. Standardize M1 dataset format (add missing fields and fix MCQ prompts)
    def standardize_m1_sample(sample):
        # Add question options to MCQ prompts if missing
        prompt = sample['prompt']
        if sample.get('question_type') == 'mcq' and sample.get('question_options'):
            if "Options:" not in prompt and sample['question_options']:
                prompt = f"{prompt}\n\nOptions:\n{sample['question_options']}"
        
        return {
            'prompt': prompt,
            'chosen': sample['chosen'],
            'rejected': sample['rejected'],
            'source': 'm1_dataset'
        }
    
    # Apply standardization to M1 dataset
    m1_standardized = m1_dpo.map(standardize_m1_sample)
    m1_standardized = m1_standardized.select_columns(['prompt', 'chosen', 'rejected', 'source'])
    
    # 4. Downsample juicy mix to manageable size
    juicy_sample_size = min(10000, len(juicy_mix))  # Sample 10k from juicy mix
    juicy_sampled = juicy_mix.shuffle(seed=42).select(range(juicy_sample_size))
    
    # Ensure juicy mix has source column
    if 'source' not in juicy_sampled.column_names:
        juicy_sampled = juicy_sampled.add_column('source', ['juicy_mix'] * len(juicy_sampled))
    
    print(f"Sampled juicy mix: {len(juicy_sampled)} samples")
    
    # 5. Combine datasets
    # Option 1: M1 only (for domain-specific training)
    m1_only = m1_standardized
    
    # Option 2: Combined dataset with balanced sampling
    # Take all M1 + sampled juicy mix
    combined_list = list(m1_standardized) + list(juicy_sampled)
    combined_dataset = Dataset.from_list(combined_list)
    
    print(f"Combined dataset: {len(combined_dataset)} samples")
    
    # 6. Create train/eval splits for both datasets
    def create_splits(dataset, test_size=0.1, seed=42):
        """Create train/eval splits"""
        dataset = dataset.shuffle(seed=seed)
        split_idx = int(len(dataset) * (1 - test_size))
        
        train_dataset = dataset.select(range(split_idx))
        eval_dataset = dataset.select(range(split_idx, len(dataset)))
        
        return DatasetDict({
            'train': train_dataset,
            'eval': eval_dataset
        })
    
    # Create splits
    m1_splits = create_splits(m1_only)
    combined_splits = create_splits(combined_dataset)
    
    print(f"M1 splits - Train: {len(m1_splits['train'])}, Eval: {len(m1_splits['eval'])}")
    print(f"Combined splits - Train: {len(combined_splits['train'])}, Eval: {len(combined_splits['eval'])}")
    
    # 7. Push to hub
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    
    # Push M1 only splits
    m1_splits.push_to_hub(f"ciacco/m1_dpo_clean_splits_{timestamp}", private=True)
    
    # Push combined splits  
    combined_splits.push_to_hub(f"ciacco/m1_dpo_combined_splits_{timestamp}", private=True)
    
    return m1_splits, combined_splits, timestamp


In [ ]:

# Run Data Pipeline
m1_splits, combined_splits, timestamp = create_clean_dpo_pipeline()


Loading datasets...
M1 dataset: 7584 samples
Juicy mix dataset: 40630 samples


Map:   0%|          | 0/7584 [00:00<?, ? examples/s]

Sampled juicy mix: 10000 samples
Combined dataset: 17584 samples
M1 splits - Train: 6825, Eval: 759
Combined splits - Train: 15825, Eval: 1759


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
